# Procedure for an indirect test of the time-drift effect:
1. Use Sloan Digital Sky Survey (SDSS) SQL Queries to select galaxies within a redshift (z) range 0.15 < z < 0.3. This range is chosen from Morseco (2011), *"Early-type galaxies as probes of galaxy formation and cosmology,"* for its S/N strength using the $D4000_n$ parameter.

2. Collect and label spectrum, mass, and feature data for each galaxy. 

    ### Spectrum data can then be used to:

    * Make data quality cuts based on error values.

    * Characterize galaxy class.

    * Determine if the galaxy has an active galactic nuclei (AGN).

    ### Make initial quality cuts 

    * For $D4000_n$, keep S/N > 2.

    * If $W_0[H\delta]_{err}$ or $W_0[OII]_{err}$ < 0: data collection error.

3. Determine the galaxy's morphology. Merger galaxies are discarded - only individual galaxies evolution will be tracked.

4. Determine the galaxy's local population density within a range of volumes from 2-48. 

5. Find the distance to the galaxy's 5th nearest neighbor (5NN), following Moresco (2011).

6.  Analysis (Discussed more in later cells)


# 1. SQL Query
This step is handled using two separate queries and results in two final CSV files with different data. The Galaxy Zoo project only has morphology estimates for 27109 galaxies in the given redshift range, while the total SDSS sample for the same range includes 324048 galaxies. To avoid timeout errors in either sample, two functions were created using the arguments of a chunk size (number of galaxies to select), the objid of the last object in the previous list (0 can be chosen for the initial value), and a general filename to query SDSS before creating a CSV file of the given chunk size number of galaxies. The two functions differ only in their query text. The generated files are named iteratively using the filename argument and the final objid in the generated table. This objid is also returned as a value to be used in a looping function. 

A second function then loops this function using the final objid until there are no galaxies left. After exhausting the pool of galaxies, this function then combines all the previous CSV files into a single larger one to simplify future function calls.

The exact RA, DEC, z, zerr, plate, mjd, and fiberid can all be collected from the FITS file, as well as the spectral data required for analysis.

This query can then return a CSV file which has as one of its columns a link to a downloadable FITS file. These files can then be downloaded and put into another file as:

In [ ]:
import pandas as pd
import requests
from pathlib import Path
from astroquery.sdss import SDSS
import time

def sdss_chunk_query(chunk_size, last_id, file_name, folder_name):
    """
    SQL search SDSS database to return a csv file with the objid, plate, mjd, fiberid,
    and FITS file URL for all galaxies between z=0.13 to z=0.3, 50000 galaxies at a time
    to prevent timeout.
    """
    sdss_chunk = f"""
SELECT TOP {chunk_size}
p.objid, s.plate, s.mjd, s.fiberid, s.z, p.ra, p.dec,
dbo.fGetUrlFitsSpectrum(s.specObjID) AS spec_fits_url
FROM PhotoObj AS p
JOIN SpecObj AS s
    ON p.objid = s.bestobjid
JOIN Galaxy AS g
    ON g.objid = p.objid
    WHERE s.class = 'GALAXY'
    AND s.z BETWEEN 0.1397816562350196 AND 0.311104966694253
    AND s.zWarning = 0
    AND p.objid > {last_id}
ORDER BY p.objid
    """
    table = SDSS.query_sql(sdss_chunk)
    if table is None:
        return None, None
    last_id = table[-1][0]
    new_file_name = f'{file_name}{last_id}.csv'
    table.write(f"{folder_name}/{new_file_name}", format="csv", overwrite=True)
    return last_id, new_file_name

def galaxy_zoo_chunk_query(chunk_size, last_id, file_name, folder_name):
    """
    SQL search SDSS database to return a csv file with the confidence rating for
    if a galaxy is elliptical, clockwise spiral, anticlockwise spiral, edgeon,
    unknown, or merger for every shared SDSS & Galaxy Zoo object between z=0.15 to z=0.3
    """
    galaxy_zoo_chunk = f"""
SELECT TOP {chunk_size}
p.objid,
zns.p_el as elliptical,
zns.p_cw as spiralclock,
zns.p_acw as spiralanticlock,
zns.p_edge as edgeon,
zns.p_dk as dontknow,
zns.p_mg as merger
FROM PhotoObj AS p
JOIN SpecObj AS s
    ON p.objid = s.bestobjid
JOIN Galaxy AS g
    ON g.objid = p.objid
JOIN ZooNoSpec AS zns
    ON zns.objid = g.objid
WHERE 
    s.class = 'GALAXY'
    AND s.z BETWEEN 0.1397816562350196 AND 0.311104966694253
    AND s.zWarning = 0
    AND p.objid > {last_id}
ORDER BY p.objid
    """
    table = SDSS.query_sql(galaxy_zoo_chunk)
    if table is None:
        return None, None
    last_id = table[-1][0]
    new_file_name = f'{file_name}{last_id}.csv'
    table.write(f"{folder_name}/{new_file_name}", format="csv", overwrite=True)
    return last_id, new_file_name

# Save and merge query data

In [ ]:
import pandas as pd
import requests
from pathlib import Path
from astroquery.sdss import SDSS
import time

def merge_csv(files, final_file, final_folder):
    """
    Take a list of CSV files and combine them into a single file.
    """
    outdir = Path(final_folder)
    outdir.mkdir(parents=True, exist_ok=True)
    df_list = [pd.read_csv(f) for f in files]
    combined = pd.concat(df_list, ignore_index=True)
    combined.to_csv(f"{outdir}/{final_file}", index=False)
    return combined

def cleanup_files(files):
    """
    Deletes a list files, helping to conserve memory.
    """
    for f in files:
        Path(f).unlink(missing_ok=True)

def loop_galaxy_chunk(query, chunk_size, last_id, file_name, final_file, folder_name):
    """
    Use the SDSS queries to create csv files up to a given chunk size and save them to a folder.
    """
    csv_file_list = []
    outdir = Path(folder_name)
    outdir.mkdir(parents=True, exist_ok=True)

    while True:
        retries = 0

        while retries <= 5:
            try:
                print(f'Collecting next {chunk_size} galaxies for {folder_name}...')
                last_id, new_file_name = query(chunk_size, last_id, file_name, folder_name)

                if last_id is None:
                    break  # exhausted — exit retry loop

                print(f'Creating builder file: {new_file_name}')
                csv_file_list.append(str(outdir / new_file_name))
                break

            except Exception as e:
                retries += 1
                if retries > 5:
                    raise
                time.sleep(3)

        if last_id is None:
            break  # exhausted — exit outer loop

    print(f'Merging {len(csv_file_list)} CSV files for {folder_name}...')
    merge_csv(csv_file_list, final_file, folder_name)
    cleanup_files(csv_file_list)
    print("Done!")

def download_fits_chunk(source_csv_file, start, end, outdir):
    """
    Uses final SDSS CSV file to fill a FITS folder with the 
    downloaded FITS files of all available galaxies in the desired redshift range (0.15-0.3).
    """
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)

    df = pd.read_csv(source_csv_file, header=0)

    for i in range(start, end):
        row = df.iloc[i]
        if 0.15 <= row['z'] <= 0.3:
        
            plate = row["plate"]
            mjd   = row["mjd"]
            fiber = row["fiberid"]
            url   = row["spec_fits_url"]

            filename = f'spec-{plate:04d}-{mjd}-{fiber:04d}.fits'
            filepath = outdir / filename

            # Handle non-existing FITS files:
            if not isinstance(url, str) or not url.strip():
                print(f"No valid url for {filename}, skipping.")
                continue

            if filepath.exists():
                continue

            r = requests.get(url, stream=True)
            r.raise_for_status()

            with open(filepath, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

# 2. Collect and label spectrum data for each galaxy
The collect_spectrum_data function is designed to extract the following data from a single FITS file, returning a dictionary of the related values.

Extracting from the returned FITS file (spec_fits_url) and converting to rest wavelength using $\lambda_{rest} = \frac{\lambda_{obs}}{(1 + z)}$:

1. Break strength at 4000 $\AA$. Flux from 4000-4100 $\AA$, $F_{red}$ and 3850-3950 $\AA$, $F_{blue}$. $$D4000_n = \frac{F_{red}}{F_{blue}}$$ This value must be calculated from the data.

2. The uncertainty in $D4000_n$, found using the inverse variance of the same flux ranges as $$\sigma(D4000_n)=D4000_n \times \sqrt{(\frac{\sigma F_{red}}{F_{red}})^2 + (\frac{\sigma F_{blue}}{F_{blue}})^2}$$ This value must be calculated from the data.

3. The flux produced by singly-ionized oxygen at a wavelength of 3727 $\AA$, denoted $[O \text{ } _{II}]\lambda3727$.

4. The rest frame equivalent width of this line ($W_0[O \text{ } _{II}]$).

5. The flux of the $H\delta$ Balmer line at 4101 $\AA$.

6. The rest frame equivalent width of this line ($W_0[H\delta]$).

In [1]:
from astropy.io import fits
import numpy as np
# Extract data

def collect_spectrum_data(file):
    """
    Compute D4000n, collect [OII] 3727 & Hdelta flux/EW from a single SDSS FITS spectrum file.

    Negative Err values = invalid fit.

    Flux is negative for absorption spectra, positive for emission.
    """
    # Open file
    hdul = fits.open(file)
    
    # Assign dfs
    hdu     = hdul[0].header
    coadd   = hdul[1].data
    specobj = hdul[2].data
    spzline = hdul[3].data

    # Info
    ra  = hdu['PLUG_RA']
    dec = hdu['PLUG_DEC']

    # Label data
    plate = spzline['PLATE'][0]
    mjd   = spzline['MJD'][0]
    fiber = spzline['FIBERID'][0]
    fileid = f'spec-{plate}-{mjd}-{fiber:04d}'
    try:
        objid = specobj['bestObjID'][0]
    except Exception as e:
        #print(f'Error on {fileid}: {e}')
        #print(f'Trying new key...')
        try:
            objid = specobj['OBJID'][0]
            #print('Success!')
        except Exception as e:
            #print(f'Failed again on {objid}: {e}')
            #print(f'Skipping object...')
            objid = None

    # Flux and wavelength
    flux       = coadd['flux']
    loglam     = coadd['loglam']
    lambda_obs = 10**loglam
    ivar       = coadd['ivar']

    # Redshift
    z     = specobj['Z'][0]
    z_err = specobj['Z_ERR'][0]

    # Shift to rest-frame
    lambda_rest = lambda_obs / (1 + z)

    # Find D4000n and uncertainty
    red_mask     = (lambda_rest >= 4000) & (lambda_rest <= 4100)
    blue_mask    = (lambda_rest >= 3850) & (lambda_rest <= 3950)
   
    F_red        = np.mean(flux[red_mask])
    red_ivar     = ivar[red_mask]

    F_blue       = np.mean(flux[blue_mask])
    blue_ivar    = ivar[blue_mask]

    D4000n       = F_red / F_blue

    # Protect against divisions by zero
    good_red = red_ivar > 0
    good_blue = blue_ivar > 0
    if good_red.sum() == 0 or good_blue.sum() == 0:
        sigma_D4000n = np.inf
    else:    
        sigma_red    = np.sqrt(1 / np.sum(red_ivar[good_red]))
        sigma_blue   = np.sqrt(1 / np.sum(blue_ivar[good_blue]))
        sigma_D4000n = D4000n * np.sqrt(
        (sigma_red / F_red)**2 +
        (sigma_blue / F_blue)**2
    )
    

    # OII 3727 Flux and EW
    oii_mask     = spzline['LINENAME']=='[O_II] 3727'
    oii_flux     = spzline['LINEAREA'][oii_mask][0]
    oii_flux_err = spzline['LINEAREA_ERR'][oii_mask][0]
    oii_EW       = spzline['LINEEW'][oii_mask][0]
    oii_EW_err   = spzline['LINEEW_ERR'][oii_mask][0]

    # H delta Flux and EW
    h_delta_mask     = spzline['LINENAME']=='H_delta'
    h_delta_flux     = spzline['LINEAREA'][h_delta_mask][0]
    h_delta_flux_err = spzline['LINEAREA_ERR'][h_delta_mask][0]
    h_delta_EW       = spzline['LINEEW'][h_delta_mask][0]
    h_delta_EW_err   = spzline['LINEEW_ERR'][h_delta_mask][0]

    # Other lines for AGN
    # OIII 5007, H beta, NII 6583, H alpha
    o_iii_mask   = spzline['LINENAME']=='[O_III] 5007'
    o_iii_flux   = spzline['LINEAREA'][o_iii_mask][0]
    h_beta_mask  = spzline['LINENAME']=='H_beta'
    h_beta_flux  = spzline['LINEAREA'][h_beta_mask][0]
    n_ii_mask    = spzline['LINENAME']=='[N_II] 6583'
    n_ii_flux    = spzline['LINEAREA'][n_ii_mask][0]
    h_alpha_mask = spzline['LINENAME']=='H_alpha'
    h_alpha_flux = spzline['LINEAREA'][h_alpha_mask][0]


    # Creating dictionary to store values
    spectrum_data_dict = {
        'objid': objid,
        'fileid': fileid,  
        'ra': ra,
        'dec': dec,                     
        'z': z,                                
        'z_err': z_err,                       
        'D4000n': D4000n, 
        'sigma_D4000n': sigma_D4000n,                    
        'oii_flux': oii_flux,                 
        'oii_flux_err': oii_flux_err,         
        'oii_EW': oii_EW,                     
        'oii_EW_err': oii_EW_err,             
        'h_delta_flux': h_delta_flux,         
        'h_delta_flux_err': h_delta_flux_err, 
        'h_delta_EW': h_delta_EW,             
        'h_delta_EW_err': h_delta_EW_err,
        'o_iii_flux': o_iii_flux,
        'h_beta_flux': h_beta_flux,
        'n_ii_flux': n_ii_flux,
        'h_alpha_flux': h_alpha_flux,    

    }

    # Return dictionary
    return spectrum_data_dict

#spectrum_data_dict = collect_spectrum_data('.\FITS\spec-417-51821-0029.fits')
#for label, value in spectrum_data_dict.items():
#    print(f"{label}: {value}")
    

# 2.1 Data quality cuts, galaxy classification.

As discussed previously, data quality cuts will be made at $\sigma(W_0[H\delta]) \ge 0.8 \text{ }\AA$. The data quality cut intended at $\sigma(D4000_n) \ge 0.03$ has proved overly restrictive for this redshift range, so another quality cut will be made based on a signal to noise (S/N) ratio of $\frac{D4000_n}{\sigma(D4000_n)}>5$. $D4000_n$ must also be positive in both bands, so negative values will be labelled as invalid. Equivalent widths (EWs) are used for S/N ratios for $W_0[H\delta]$ and $W_0[O\text{ }II]$. Detections for these bands are defined as $|EW|/EW_{err} \ge 2$, otherwise the line will be considered "Absent."

For characterizing SFH:

* $D4000_n$ tracks long-term, light-weighted (meaning weighted by how much light the stellar population contributes to the spectrum as opposed to mass-weighted) stellar age. Larger values $\to$ older.

* $W_0[H\delta]$ is sensitive to **recent changes** (~0.1-1 Gyr) in star formation. Stronger absorption $\to$ formation stopped or declined more recently.

* $W_0[O \text{ }_{II}]$ relates to recent star formation ($\le 10-50$ Myr).

Following Poggianti et al. (1999), *"The star formation histories of galaxies in distant clusters,"* the EWs can be used to classify the galaxies. The table from Poggianti et al. goes as:

| Class | $W_0[O\text{ }_{II}] \lambda 3727$ ($\AA$) | $W_0[H\delta]$ ($\AA$) | Comments                                                    |
| ---   | :---:                                      | :---:                  | ---                                                         |
|k      |Absent                                      |<3                      |Passive                                                      |
|k+a    |Absent                                      |3-8                     |Moderate Balmer absorption without emission                  |
|a+k    |Absent                                      |$\ge 8$                 |Strong Balmer absorption without emission                    |
|e(c)   |Yes, $\lt 40$                               |$\lt 4$                 |Moderate Balmer absorption plus emission, spiral-like        |
|e(a)   |Yes                                         |$\ge 4$                 |Strong Balmer absorption plus emission                       |
|e(b)   |$\ge 40$                                    |...                     |Starburst                                                    |
|e(n)   |...                                         |...                     |AGN from broad lines or $[O\text{ }_{II}] 5007/H\beta$ ratio |
|e      |Yes                                         |?                       |At least one emission line, but S/N too low to classify      |
|?      |?                                           |?                       |Unclassifiable                                               |

"Absent" here is classified as an EW stronger than -5 $\AA$ (meaning <-5). The "..." entries indicate that the specific spectral index is not used to define the galaxy class. For Active Galactic Nuclei (AGN) objects, as both EWs for [O II] and $H\delta$ are not used, this paper will use the definition of an AGN as given by Kauffmann et al. (2003), *"The Host Galaxies of AGN,"* using instead the [O III], $H\beta$, [N II], and $H\alpha$ lines. By their definition, a galaxy is defined as an AGN if
$$log(\frac{[O\text{ }III]}{H\beta}) > 0.61 [log(\frac{[N\text{ }II]}{H\alpha})-0.05]^{-1}+1.3$$

The letters in the classification denote the dominant spectral types (k $\to$ older K-type stars, a $\to$ A-type stars, e $\to$ emission lines $\to$ ongoing star formation). So:

| Class |Dominant star description                                                                                                       |
|---    | ---                                                                                                                            |
|k      |K-type. Quenched, passively evolving galaxy. No emission lines.                                                                 |
|k+a    |Mainly K-type, some A-type. Recently ended star formation.                                                                      |
|a+k    |Mainly A-type, some K-type. More extreme post-starburst.                                                                        |
|e(c)   |"Continuous." Star-forming galaxy, typical of spirals galaxies.                                                                 |
|e(a)   |Star-forming + strong A-type signature. Recent burst or rapid decline.                                                          |
|e(b)   |"Burst." Ongoing starburst, dominated by younger O/B-type stars.                                                                |
|e(n)   |AGN. Emission does not trace star formation.                                                                                    |
|e      |Emission present but unclassifiable.                                                                                            |
|?      |Data quality too poor to assign class. $\|EW\|/EW_{err} \lt 2$ for both $H\delta$ and $[O\text{ }_{II}]$ and\or $D4000_n \lt 0$ |

Galaxy classes will be assigned following the format of Poggianti et al. (1999). ? is reserved for galaxies which fail the quality cuts.

For the purposes of this thesis, galaxy classification will be used to relate the local population density of the galaxy to the type of galaxy. If a disproportionate number of k-type galaxies are in voids and a disproportionate number of e(c), e(b), and/or e(a) galaxies are in clusters, these results would be in favor of timescape cosmology.

# 2.2 Galaxy Morphology
As a citizen science project, Galaxy Zoo offers estimates for the morphology of a galaxy in the manner of votes provided by users of different possible shapes. The possible votes for a galaxy are spiral - either clockwise or anticlockwise, elliptical, edge on (disk), merger, and unknown ("dontknow"). In the determine_shape function, a galaxy is only counted as being one of the given shapes if the votes for that shape account for twice as many as the next highest voted shape, or S/N > 2. Galaxies that do not pass this S/N threshold (and galaxies that pass this threshold as "dontknow") will be returned as "dontknow" and will not be used for analysis. This function uses the objid to find the galaxy and is paired with a future function (sort_values) and the sort_galaxy function to fetch the shape and classification at the same time for any galaxy.

In [ ]:
import csv

def sort_galaxy(spectrum_data_dict):
    """
    Use spectrum values to determine the galaxy's spectral class.
    """
    # Assign variables
    h_delta_EW     = spectrum_data_dict['h_delta_EW']
    h_delta_EW_err = spectrum_data_dict['h_delta_EW_err']
    oii_EW         = spectrum_data_dict['oii_EW']
    oii_EW_err     = spectrum_data_dict['oii_EW_err']
    D4000n         = spectrum_data_dict['D4000n']
    sigma_D4000n   = spectrum_data_dict['sigma_D4000n']
    o_iii          = spectrum_data_dict['o_iii_flux']
    h_beta         = spectrum_data_dict['h_beta_flux']
    n_ii           = spectrum_data_dict['n_ii_flux']
    h_alpha        = spectrum_data_dict['h_alpha_flux']

    # Quality cuts
    if (h_delta_EW_err < 0) or (oii_EW_err < 0):
        return '?: Invalid EW value'
    elif  (D4000n / sigma_D4000n) < 2:
        return '?: D4000n quality cut'
    
    # AGN before other classes
    # Avoiding division by zero/require positive values
    if (o_iii > 0) and (h_beta > 0) and (n_ii > 0) and (h_alpha > 0):

        x = np.log10(n_ii/h_alpha)
        y = np.log10(o_iii/h_beta)
        # Guard against vertical asymptote
        if not np.isclose(x - 0.05, 0.0):
            if y > (0.61 / (x - 0.05) + 1.3):
                return 'e(n)'
    
    # OII W_0 "Absent"
    if (oii_EW < -5):
        if h_delta_EW < 3:
            return 'k'
        elif (3 < h_delta_EW < 8):
            return 'k+a'
        elif h_delta_EW >= 8:
            return 'a+k'
        
    # OII EW present
    elif (oii_EW > -5):
        if (oii_EW < 40) and (h_delta_EW < 4):
            return 'e(c)'
        elif (oii_EW >= 40):
            return 'e(b)'
        elif h_delta_EW >=4:
            return 'e(a)'
        else:
            return 'e'

def determine_shape(objid, file_path="ZOO/full_morphology"):
    # Look up if objid is in full_morphology.csv
    file_path = Path(file_path)
    with open(file_path, newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if int(row['objid']) == objid:
                valid_row = row
                break
        else:
            return None # objid not found
        
    shapes = {
        k: float(v)
        for k, v in valid_row.items()
        if k != 'objid'
    }

    # Sort shapes
    sorted_shapes = sorted(
        shapes.items(),
        key=lambda item: item[1],
        reverse=True
    )

    # Find S/N ratio
    (shape1, val1), (shape2, val2) = sorted_shapes[:2]
    # Reject shape2 automatically if val2 is 0
    if val2 == 0:
        return shape1
    
    ratio = val1 / val2
    if ratio >= 2:
        return shape1
    else:
        return "dontknow"


shape = determine_shape(1237648702966464576, "ZOO/full_morphology.csv")
print(shape)
    

elliptical


The following code cell is where the data sorting takes place. First, it is necessary to create a function that can count the number of galaxies around a chosen object at different radii (2, 5, 10, and 15 Mpc) by performing a new SDSS SQL query. To do this requires using the relationship between the angular diameter, $\theta$, in units of radians, the distance from the object, $d_A$, and the length of the object, $x$. This is the **angular diameter distance**, and can be found as 
$$d_A = \frac{x}{\theta}$$
This relationship becomes complicated under $\Lambda \text{CDM}$ at redshifts greater than about 1.5, at which point objects begin to appear larger with increasing redshift, but this domain is unimportant for redshift range being investigated. Under an assumption of Euclidean geometry, the relationship between size and distance is 
$$tan(\theta)=\frac{x}{d_A}$$

The astropy.cosmology package for Python is able to convert any given redshift value to a distance value (using an assumption of $\Lambda \text{CDM}$ cosmology), so a simple function can be written to use that value as an input along with a chosen radius value to determine the search radius for the SQL query.

This creates a search cone which returns objects which are **visually** nearby the galaxy on the sky but may be arbitrarily close or far away from the galaxy in physical space. To select only the objects which are physically relevant to the chosen galaxy means also ensuring that objects are within a given redshift related to the search radius. Along the line of sight, distances relate to redshift via the physical distance $\Chi_{phys}$ as: 
$$\frac{d\Chi_{phys}}{dz} = \frac{c}{H(z)}$$
Meaning for a small distance $\Delta\Chi_{phys}$: 
$$\Delta z \approx \frac{\Delta\Chi_{phys}}{d\Chi_{phys}/dz}=\Delta\Chi_{phys} \frac{H(z)}{c}$$
Where $H(z)$ is the Hubble parameter at redshift $z$, and $c$ is the speed of light.
These values can then be used to determine the total volume of the search area using the equation for a sliced cone (frustrum) $$V = \frac{1}{3}\pi h(R_1^2 + R_1 R_2 + R_2^2)$$
where $h$ is the height/depth of the frustrum, $R_1$ is the radius of the larger end, and $R_2$ is the radius of the smaller top. The radii values can be found again using $R = tan(\theta)\cdot d_A$ relationship, with $R_1$ using the distance to the far end of the frustrum found by $z + \Delta z$ while $R_2$ would instead use $z-\Delta z$.
The query then only needs to use the calculated values to count the number of objects in the search window within a redshift range before collecting the values for later use. 


In [118]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
from astropy.cosmology import Planck18 as cosmo
import astropy.units as u
import astropy.constants as const
from astroquery.sdss import SDSS
from scipy.spatial import cKDTree
from astropy.coordinates import SkyCoord
import pickle
import matplotlib.pyplot as plt
import numpy as np
import os

#-----------------------------------------------
# Transform search radius into theta and delta z
#-----------------------------------------------

def length_to_radius(z, mpc_radius, dis_type):
    """
    Convert a radius (Mpc) at a given redshift to an angular radius in arcminutes
    using either proper distance or comoving distance values.
    Use the redshift to return the distance to the object in Mpc as well.
    """
    # Convert z to Mpc
    if dis_type == 'proper':
        distance = cosmo.angular_diameter_distance(z)
    if dis_type == 'comoving':
        distance = cosmo.comoving_distance(z)
    # Calculate diameter theta
    theta           = np.arctan((mpc_radius * u.Mpc) / distance)
    # Convert to arcmin and radius
    radius_arcmin   = theta.to(u.arcmin).value
    return radius_arcmin

def length_to_delta_z(z, mpc_radius, dis_type):
    """
    Given a redshift and search window, find the delta z using either proper or comoving distance.
    """
    # Hubble parameter at redshift z
    if dis_type == 'proper':
        r = mpc_radius * u.Mpc
    if dis_type == 'comoving':
        r = mpc_radius * (1 + z) * u.Mpc
    Hz = cosmo.H(z) # km / s / Mpc

    # Convert H(z)/c to 1/distance units
    c = const.c.to(u.km/u.s)
    delta_z = ((Hz / c) * r).decompose()

    return delta_z.value

#-------------------------------------------------------
# Count number of neighbors and calculate number density
#-------------------------------------------------------

def count_sdss_neighbors(data_dict, mpc_radius, dis_type, ra_all, dec_all, z_all):
    ra0  = data_dict['ra']
    dec0 = data_dict['dec']
    z0   = data_dict['z']

    # Angular radius (arcmin)
    radius_arcmin = length_to_radius(z0, mpc_radius, dis_type)

    # Redshift window
    dz = length_to_delta_z(z0, mpc_radius, dis_type)
    z_min = z0 - dz
    z_max = z0 + dz

    # Redshift filter boolean mask
    z_mask = (z_all >= z_min) & (z_all <= z_max)

    # Angular separation
    dra = (ra_all - ra0) * np.cos(np.deg2rad(dec0))
    ddec = dec_all - dec0
    ang_sep_deg = np.sqrt(dra ** 2 + ddec ** 2)
    ang_sep_arcmin = ang_sep_deg * 60

    ang_mask = ang_sep_arcmin < radius_arcmin

    total_mask = z_mask & ang_mask

    count = np.sum(total_mask)

    # Subtract self if included
    return max(count - 1, 0)

def cone_slice_volume_calculator(z, mpc_radius, dis_type):
    """
    Calculate the volume of the search area using the equation of a frustrum.
    """
    if dis_type == 'proper':
        distance_center = cosmo.angular_diameter_distance(z)
    if dis_type == 'comoving':
        distance_center = cosmo.comoving_distance(z)

    theta = np.arctan((mpc_radius * u.Mpc) / distance_center) # keep in radians

    # Half-width along line of sight in redshift
    delta_z = length_to_delta_z(z, mpc_radius, dis_type)

    # Distance to near and far planes
    if dis_type == 'proper':
        near_distance = cosmo.angular_diameter_distance(z-delta_z).value
        far_distance = cosmo.angular_diameter_distance(z+delta_z).value
    if dis_type == 'comoving':
        near_distance = cosmo.comoving_distance(z-delta_z).value
        far_distance = cosmo.comoving_distance(z+delta_z).value

    # Calculate near and far radii
    r_near = np.tan(theta) * near_distance
    r_far  = np.tan(theta) * far_distance

    h = far_distance-near_distance

    return (1/3) * np.pi * h * (r_near ** 2 + (r_near * r_far) + r_far **2)

def calculate_density(n_neighbors, volume):
    """
    Use a galaxy's number of neighbors to calculate the number density for a given volume.
    """
    return n_neighbors / volume

#------------------------------------------------
# Find 5NN for both proper and comoving distances
#------------------------------------------------

def find_fifth_nearest_neighbor(ra_all, dec_all, z_all):
    """
    Use cKDTree to build a tree out of the sdss csv file of the distances between galaxies. Find the distance to the fifth nearest neighbor (5NN)
    using both the physical and the comoving distance
    """
    # Find 5NN using proper distance
    physical_distance_all = cosmo.angular_diameter_distance(z_all) # Mpc
    coords_from_phys = SkyCoord(ra=ra_all*u.deg, dec=dec_all*u.deg, distance=physical_distance_all)
    xyz_phys = np.vstack(coords_from_phys.cartesian.xyz).T
    tree_phys = cKDTree(xyz_phys)
    dis_phys, _ = tree_phys.query(xyz_phys, k=6)
    fifth_phys = dis_phys[:, 5]

    # Find 5NN using comoving distance
    comoving_distance_all = cosmo.comoving_distance(z_all) # Mpc
    coords_from_comv = SkyCoord(ra=ra_all*u.deg, dec=dec_all*u.deg, distance=comoving_distance_all)
    xyz_comv = np.vstack(coords_from_comv.cartesian.xyz).T
    tree_comv = cKDTree(xyz_comv)
    dis_comv, _ = tree_comv.query(xyz_comv, k=6)
    fifth_comv = dis_comv[:, 5]

    return fifth_phys, fifth_comv

#-------------------------------------------------------------------
# Collect all relevant values and load them into a result dictionary
#-------------------------------------------------------------------

def collect_values(files, csv_file_path):
    """
    Using the FITS files, store the objid, redshift, D4000n, sigma D4000n, Hdelta EW, Hdelta err, oii EW, oii EW err, 
    number density (for 2, 5, 10, 15, 21, and 42 Mpc search windows), galaxy class, and galaxy shape (if available)
    for every galaxy available. 
    """
    # Load full_sdss.csv for neighbor counting
    volume_df = pd.read_csv(csv_file_path, dtype={'objid': str})
    ra_all    = volume_df['ra'].values
    dec_all   = volume_df['dec'].values
    z_all     = volume_df['z'].values
    objid_all = volume_df['objid'].astype(str).values
    # Collect the index of every object for recall
    objid_to_index = {str(objid): i for i, objid in enumerate(objid_all)}    

    # Load every galaxy's 5NN
    fifth_phys_all, fifth_comv_all = find_fifth_nearest_neighbor(ra_all, dec_all, z_all)

    # Initiate type dictionaries
    class_dict = defaultdict(list)
    shape_dict = defaultdict(list)

    # Radii to calculate neighbors for
    mpc_radii = [2, 5, 10, 15, 21, 42] # cluster core to typical void radius

    # Collect values
    for file in files:
        spectrum_data_dict = collect_spectrum_data(file)
        objid = spectrum_data_dict['objid']

        # Only select data in the chosen redshift range
        z = spectrum_data_dict['z']
        if (z < 0.15) or (z > 0.3):
            continue
        
        # Skip missing objids
        if objid == None:
            print(f"Failed to find objid for {spectrum_data_dict['fileid']}.")
            continue
        if objid not in objid_to_index:
            print(f"Failed to find objid {objid} in csv file (from {spectrum_data_dict['fileid']})")
            continue
        
        idx = objid_to_index[objid]

        galaxy_class = sort_galaxy(spectrum_data_dict)
        galaxy_shape = determine_shape(objid, "ZOO/full_morphology.csv")

        # Grab proper and comoving distances from 5NN
        fifth_nn_proper = fifth_phys_all[idx]
        fifth_nn_comv = fifth_comv_all[idx]

        # Calcuate neighbor counts and density for each radius using proper and comoving distances
        proper_n_neighbors = []
        proper_densities   = []

        comoving_n_neighbors = []
        comoving_densities   = []
        for r in mpc_radii:
            #-------------------------------
            # Collect proper distance values
            #-------------------------------
            proper_nn_count = count_sdss_neighbors(spectrum_data_dict, r, 'proper', ra_all, dec_all, z_all)
            proper_n_neighbors.append(proper_nn_count)

            # Compute frustrum volume for this radius
            proper_volume = cone_slice_volume_calculator(spectrum_data_dict['z'], r, 'proper')
            proper_densities.append(calculate_density(proper_nn_count, proper_volume))

            #---------------------------------
            # Collect comoving distance values
            #---------------------------------
            comoving_nn_count = count_sdss_neighbors(spectrum_data_dict, r, 'comoving', ra_all, dec_all, z_all)
            comoving_n_neighbors.append(comoving_nn_count)

            # Compute frustrum volume for this radius
            comoving_volume = cone_slice_volume_calculator(spectrum_data_dict['z'], r, 'comoving')
            comoving_densities.append(calculate_density(comoving_nn_count, comoving_volume))

        # Store all data to sort by galaxy class
        class_dict[galaxy_class].append(
            {
                'objid': spectrum_data_dict['objid'],
                'z': spectrum_data_dict['z'],
                'ra': spectrum_data_dict['ra'],
                'dec': spectrum_data_dict['dec'],
                'D4000n': spectrum_data_dict['D4000n'], 
                'sigma_D4000n': spectrum_data_dict['sigma_D4000n'],
                'h_delta_EW': spectrum_data_dict['h_delta_EW'], 
                'h_delta_EW_err': spectrum_data_dict['h_delta_EW_err'],
                'oii_EW': spectrum_data_dict['oii_EW'],
                'oii_EW_err': spectrum_data_dict['oii_EW_err'],
                'proper_densities': proper_densities,
                'comoving_densities': comoving_densities,
                'fifth_nn_proper': fifth_nn_proper,
                'fifth_nn_comv': fifth_nn_comv,
                'galaxy_shape': galaxy_shape
            }
        )

        # Store all data to sort by galaxy shape
        shape_dict[galaxy_shape].append(
            {
                'objid': spectrum_data_dict['objid'],
                'ra': spectrum_data_dict['ra'],
                'dec': spectrum_data_dict['dec'],
                'z': spectrum_data_dict['z'],
                'ra': spectrum_data_dict['ra'],
                'dec': spectrum_data_dict['dec'],
                'D4000n': spectrum_data_dict['D4000n'], 
                'sigma_D4000n': spectrum_data_dict['sigma_D4000n'],
                'h_delta_EW': spectrum_data_dict['h_delta_EW'], 
                'h_delta_EW_err': spectrum_data_dict['h_delta_EW_err'],
                'oii_EW': spectrum_data_dict['oii_EW'],
                'oii_EW_err': spectrum_data_dict['oii_EW_err'],
                'proper_densities': proper_densities,
                'comoving_densities': comoving_densities,
                'fifth_nn_proper': fifth_nn_proper,
                'fifth_nn_comv': fifth_nn_comv,
                'galaxy_class': galaxy_class
            }
        )

    result = {
        'class_dict': class_dict,
        'shape_dict': shape_dict
    }

    return result

def load_result(filename):
    """
    Load pickle collect_values result after saving with save_result
    """
    with open(filename, 'rb') as f:
        return pickle.load(f)

def save_result(result, filename):
    """
    Quickly save result to disk after running collect_values to avoid running multiple times
    """
    with open(filename, 'wb') as f:
        pickle.dump(result, f)

def save_job_pickle(source_folder, out_folder, start, end):
    files = sorted(Path(source_folder).iterdir())
    files = files[start:end]
    result = collect_values(files, "SDSS/full_sdss.csv")

    filename = f'pickle_{start}_{end}.pkl'

    out_path = Path(out_folder)
    out_path.mkdir(parents=True, exist_ok=True)   # <-- create folder if needed

    path = out_path / filename

    save_result(result, path)

def merge_pickles(source_folder, filename, out_folder):
    out_path = Path(out_folder)
    out_path.mkdir(parents=True, exist_ok=True)
    path = out_path / filename

    merged_class_dict = defaultdict(list)
    merged_shape_dict = defaultdict(list)

    for file in Path(source_folder).glob("pickle_*"):
        result = load_result(file)

        # Merge class_dict
        for galaxy_class, entries in result['class_dict'].items():
            merged_class_dict[galaxy_class].extend(entries)

        # Merge shape_dict
        for galaxy_shape, entries in result['shape_dict'].items():
            merged_shape_dict[galaxy_shape].extend(entries)

    result = {
        'class_dict': merged_class_dict,
        'shape_dict': merged_shape_dict
    }

    save_result(result, path)

    return result

# Finished collecting data!
Now that data has been collected, the result dictionary is loaded and relevant quality/completeness cuts are made. This is done in the `data_loading_notebook` where the final analysis sample is created.